In [30]:
import os 
import numpy as np
import scipy.io as sio
from scipy.signal import butter,filtfilt

#the bandpass filter
#4Hz-38Hz
def apply_bandpass(data, lowcut= 4.0, highcut = 38.0, fs = 250.0, order = 4):
    nyq = 0.5*fs
    low = lowcut/nyq
    high = highcut/nyq
    b, a = butter(order, [low, high], btype = 'band')

    return filtfilt(b,a,data,axis = 0)

def process_and_save_subject(mat_file_path, subject_id):
    print(f"\nProcessing subject {subject_id}")
    mat_data = sio.loadmat(mat_file_path)
    all_runs = mat_data['data'][0]

    SFREQ = 250
    Action_start = int(0.5*SFREQ)
    Action_end = int(2.5*SFREQ)
    Baseline_start = int(-0.2*SFREQ)
    baseline_end = 0

    all_clean_epochs = []
    all_clean_labels = []

    for run_idx, run in enumerate(all_runs):
        X = run['X'].item()
        y = run['y'].item()
        trial = run['trial'].item()

        if len(y) == 0:
            continue #no actual data present because non-motor stuff

        eeg_only = X[:, :22]
        filtered_eeg = apply_bandpass(eeg_only)

        for i in range(len(trial)):
            start_idx = trial[i][0]
            epoch = filtered_eeg[start_idx + Action_start : start_idx + Action_end , :]

            baseline = filtered_eeg[start_idx + Baseline_start : start_idx + baseline_end , :]

            b_mean = np.mean(baseline, axis = 0)
            b_std = np.std(baseline, axis = 0)
            normalized_epoch = (epoch-b_mean) / (b_std + 1e-6)
            
            if np.max(np.abs(normalized_epoch)) < 100:
                all_clean_epochs.append(normalized_epoch)
                all_clean_labels.append(y[i][0] - 1)

    final_X = np.array(all_clean_epochs).transpose(0, 2, 1)
    final_y = np.array(all_clean_labels)

    print(f"Extracted {len(final_y)} clean trials.")
    print(f"Final X Tensor Shape: {final_X.shape}")
    
    # --- 3. Save to Disk ---
    save_dir = "processed_data"
    os.makedirs(save_dir, exist_ok=True)
    
    np.save(os.path.join(save_dir, f"sub_{subject_id}_X.npy"), final_X)
    np.save(os.path.join(save_dir, f"sub_{subject_id}_y.npy"), final_y)
    print(f"Saved to {save_dir}/sub_{subject_id}_X.npy")

# Run it for Subject 1
process_and_save_subject('TrainingTesting-Data/A09T.mat', '09')


Processing subject 09
Extracted 288 clean trials.
Final X Tensor Shape: (288, 22, 500)
Saved to processed_data/sub_09_X.npy


In [31]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class BCIDataset(Dataset):
    def __init__(self, x_path, y_path):
        self.X = np.load(x_path)
        self.Y = np.load(y_path)

        self.X = torch.tensor(self.X, dtype = torch.float32)
        self.Y = torch.tensor(self.Y, dtype = torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx): 
        return self.X[idx], self.Y[idx]

dataset = BCIDataset('processed_data/sub_09_X.npy', 'processed_data/sub_09_y.npy')
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)  

data_batch, label_batch = next(iter(dataloader))
print(f"Batch X Shape: {data_batch.shape} -> (Batch_Size, Channels, Time)")
print(f"Batch y Shape: {label_batch.shape} -> (Batch_Size)")

Batch X Shape: torch.Size([32, 22, 500]) -> (Batch_Size, Channels, Time)
Batch y Shape: torch.Size([32]) -> (Batch_Size)


In [32]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class BCIDataset(Dataset):
    def __init__(self, x_path, y_path):
        self.X = np.load(x_path)
        self.Y = np.load(y_path)

        self.X = torch.tensor(self.X, dtype = torch.float32)
        self.Y = torch.tensor(self.Y, dtype = torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx): 
        return self.X[idx], self.Y[idx]

dataset = BCIDataset('processed_data/sub_09_X.npy', 'processed_data/sub_09_y.npy')
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)  

data_batch, label_batch = next(iter(dataloader))
print(f"Batch X Shape: {data_batch.shape} -> (Batch_Size, Channels, Time)")
print(f"Batch y Shape: {label_batch.shape} -> (Batch_Size)")

Batch X Shape: torch.Size([32, 22, 500]) -> (Batch_Size, Channels, Time)
Batch y Shape: torch.Size([32]) -> (Batch_Size)


In [ ]:
import torch.nn as nn

class SupervisedSpatialGatedCAE(nn.Module):
    def __init__(self, in_channels = 22, virtual_channels=8, latent_dim = 16, num_classes=4):
        super(SupervisedSpatialGatedCAE, self).__init__()
        self.spatial_gate = nn.Conv1d(in_channels, virtual_channels, kernel_size=1)

        self.temporal_blur = nn.MaxPool1d(kernel_size = 4)
        flattened_size = virtual_channels * 125
    
        self.encoder = nn.Sequential(
            nn.Linear(flattened_size, 64),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Dropout(0.5), #changes the weights being trained to prevent overfitting 
            nn.Linear(64, latent_dim), #linear transformation of data
            nn.LeakyReLU(negative_slope=0.01) #normaliser
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Linear(64, flattened_size)
        )

        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, num_classes)
        )
    def forward(self, x):
        mixed_x = self.spatial_gate(x) #the bottleneck part
        blurred_x = self.temporal_blur(mixed_x) #blurs the time to handle power changes and reduce errors
        flattened = blurred_x.view(blurred_x.size(0),-1)
        latent_weights = self.encoder(flattened)
        reconstructed = self.decoder(latent_weights)
        class_predictions = self.classifier(latent_weights)

        return reconstructed, latent_weights, flattened, class_predictions
    
model = SupervisedSpatialGatedCAE()

sample_X, sample_y = next(iter(dataloader))

reconstructed, latent, flattened, class_preds = model(sample_X)

print(f"Class Predictions Shape: {class_preds.shape} -> Should be (Batch_Size, 4)")


Class Predictions Shape: torch.Size([32, 4]) -> Should be (Batch_Size, 4)


In [34]:
#Loss calculator
def compute_contractive_loss(reconstructed, original, latent_weights, model, lambda_reg):
    mse_loss = nn.MSELoss()(reconstructed, original)

    W = model.encoder[3].weight
    dh = latent_weights * (1-latent_weights)

   # (Sensitivity of weights) * (State of the activation gate)
    jacobian_penalty = torch.sum(torch.pow(dh,2) * torch.sum(torch.pow(W,2),dim=1), dim=1) 

    #lambda_reg: How aggressively we punish noise
    
    return mse_loss + (lambda_reg*jacobian_penalty.mean())

In [35]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np


print("--- LOADING AND SPLITTING DATA ---")
X_full = np.load('processed_data/sub_09_X.npy')
y_full = np.load('processed_data/sub_09_y.npy')
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42, stratify=y_full)

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# Create two separate DataLoaders
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=32, shuffle=False)

print(f"Training on {len(y_train)} trials. Testing blindly on {len(y_test)} trials.")

--- LOADING AND SPLITTING DATA ---
Training on 230 trials. Testing blindly on 58 trials.


In [46]:
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
supervised_model = SupervisedSpatialGatedCAE().to(device)

optimizer = optim.AdamW(supervised_model.parameters(), lr=0.001, weight_decay=1e-2)
classification_criterion = nn.CrossEntropyLoss() #cross domain loss function

EPOCHS = 50
print(f"--- STARTING SUPERVISED TRAINING ON: {device} ---")

for epoch in range(EPOCHS):
    supervised_model.train()
    running_total_loss = 0.0
    running_class_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    for batch_X, batch_y in dataloader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device) 
        
        optimizer.zero_grad()
        
        # Forward Pass (The Y-Split)
        reconstructed, latent, flattened_original, class_preds = supervised_model(batch_X)
        
        #reconstruction loss
        #pass `supervised_model` here so the Jacobian penalty checks the right weights, passing only the CAE weights
        loss_contractive = compute_contractive_loss(reconstructed, flattened_original, latent, supervised_model, 1e-4)
        
        # classification errors, passing the class_preds weights 
        loss_classification = classification_criterion(class_preds, batch_y)
        
        #combining both losses
        total_loss = loss_classification + loss_contractive
        
        # Backpropagation
        total_loss.backward()
        optimizer.step()
        
        # Tracking metrics for display
        running_total_loss += total_loss.item()
        running_class_loss += loss_classification.item()
        
        # Calculate training accuracy for this batch
        # torch.max returns the value and the index. The index (0,1,2,3) is our predicted class.
        _, predicted_labels = torch.max(class_preds, 1)
        correct_predictions += (predicted_labels == batch_y).sum().item()
        total_samples += batch_y.size(0)
        
    if (epoch + 1) % 10 == 0:
        avg_tot_loss = running_total_loss / len(dataloader)
        avg_cls_loss = running_class_loss / len(dataloader)
        train_acc = (correct_predictions / total_samples) * 100
        print(f"Epoch [{epoch + 1}/{EPOCHS}] | Class Loss: {avg_cls_loss:.4f} | Total Loss: {avg_tot_loss:.4f} | Train Acc: {train_acc:.2f}%")

print("\nSUPERVISED TRAINING COMPLETE")


--- STARTING SUPERVISED TRAINING ON: cuda ---
Epoch [10/50] | Class Loss: 1.2251 | Total Loss: 1.2748 | Train Acc: 63.19%
Epoch [20/50] | Class Loss: 0.7947 | Total Loss: 0.8729 | Train Acc: 86.11%
Epoch [30/50] | Class Loss: 0.4616 | Total Loss: 0.5459 | Train Acc: 97.22%
Epoch [40/50] | Class Loss: 0.2775 | Total Loss: 0.3557 | Train Acc: 99.65%
Epoch [50/50] | Class Loss: 0.1689 | Total Loss: 0.2370 | Train Acc: 100.00%

SUPERVISED TRAINING COMPLETE


In [47]:
# --- BOOTING THE UPDATED MODEL ---
# MAKE SURE YOU CHANGED nn.Sigmoid() to nn.LeakyReLU() IN YOUR CLASS DEFINITION!
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SupervisedSpatialGatedCAE().to(device)

# We lower the learning rate slightly so it doesn't jump over the solution
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

print("\n--- STARTING PURE CLASSIFICATION TRAINING ---")
EPOCHS = 75 # Giving it a bit more time to learn slowly
for epoch in range(EPOCHS):
    model.train()
    running_class_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device, dtype=torch.float32), batch_y.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        _, _, _, class_preds = model(batch_X)
        
        # WE ONLY CARE ABOUT GUESSING THE BODY PART NOW
        loss_class = criterion(class_preds, batch_y)
        
        loss_class.backward()
        optimizer.step()
        
        running_class_loss += loss_class.item()
        
    if (epoch + 1) % 15 == 0:
        avg_cls_loss = running_class_loss / len(train_loader)
        print(f"Epoch [{epoch + 1}/{EPOCHS}] | Class Loss: {avg_cls_loss:.4f}")

# --- THE BLIND TEST ---
print("\n--- FINAL EVALUATION ON UNSEEN DATA ---")
model = model.to(device)
model.eval()
all_true = []
all_preds = []

if torch.cuda.is_available():
    torch.cuda.empty_cache()

with torch.no_grad():
    for batch_X, batch_y in test_loader: 
        batch_X = batch_X.to(device, dtype=torch.float32)
        _, _, _, class_preds = model(batch_X)
        _, predicted = torch.max(class_preds, 1)
        
        all_true.extend(batch_y.numpy())
        all_preds.extend(predicted.cpu().numpy())

accuracy = accuracy_score(all_true, all_preds)
print(f"True Generalization Accuracy: {accuracy * 100:.2f}%\n")
print(confusion_matrix(all_true, all_preds))


--- STARTING PURE CLASSIFICATION TRAINING ---
Epoch [15/75] | Class Loss: 1.1572
Epoch [30/75] | Class Loss: 0.8332
Epoch [45/75] | Class Loss: 0.5697
Epoch [60/75] | Class Loss: 0.3633
Epoch [75/75] | Class Loss: 0.2683

--- FINAL EVALUATION ON UNSEEN DATA ---
True Generalization Accuracy: 29.31%

[[4 5 3 3]
 [6 3 1 4]
 [2 3 8 2]
 [6 2 4 2]]


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SupervisedSpatialGatedCAE().to(device)

optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

alpha = 0.85  # 85% Priority: Intent decoding
beta = 0.15   # 15% Priority: Wave reconstruction

print("\nSTARTING BALANCED MULTI-TASK TRAINING")
EPOCHS = 75 
for epoch in range(EPOCHS):
    model.train()
    running_tot_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device, dtype=torch.float32), batch_y.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass (Getting both reconstruction and class predictions)
        reconstructed, latent, flattened, class_preds = model(batch_X)
        
        # Calculate both losses
        loss_class = criterion(class_preds, batch_y)
        loss_contractive = compute_contractive_loss(reconstructed, flattened, latent, model, 1e-4)
        
        # MULTI-TASK WEIGHTING
        total_loss = (alpha * loss_class) + (beta * loss_contractive)
        
        total_loss.backward()
        optimizer.step()
        
        running_tot_loss += total_loss.item()
        
    if (epoch + 1) % 15 == 0:
        avg_tot_loss = running_tot_loss / len(train_loader)
        print(f"Epoch [{epoch + 1}/{EPOCHS}] | Weighted Total Loss: {avg_tot_loss:.4f}")

# --- THE BLIND TEST ---
print("\nFINAL EVALUATION ON UNSEEN DATA")
model = model.to(device)
model.eval()
all_true = []
all_preds = []

if torch.cuda.is_available():
    torch.cuda.empty_cache()

with torch.no_grad():
    for batch_X, batch_y in test_loader: 
        batch_X = batch_X.to(device, dtype=torch.float32)
        _, _, _, class_preds = model(batch_X)
        _, predicted = torch.max(class_preds, 1)
        
        all_true.extend(batch_y.numpy())
        all_preds.extend(predicted.cpu().numpy())

accuracy = accuracy_score(all_true, all_preds)
print(f"Final Generalization Accuracy: {accuracy * 100:.2f}%\n")
print("Confusion Matrix:")
print(confusion_matrix(all_true, all_preds))


--- STARTING BALANCED MULTI-TASK TRAINING ---
Epoch [15/75] | Weighted Total Loss: 1.1342
Epoch [30/75] | Weighted Total Loss: 0.9428
Epoch [45/75] | Weighted Total Loss: 0.6926
Epoch [60/75] | Weighted Total Loss: 0.5067
Epoch [75/75] | Weighted Total Loss: 0.3756

--- FINAL EVALUATION ON UNSEEN DATA ---
Final Generalization Accuracy: 27.59%

Confusion Matrix:
[[4 5 3 3]
 [5 4 2 3]
 [5 1 5 4]
 [2 4 5 3]]


In [ ]:
print("\nFINAL EVALUATION ON UNSEEN DATA")

model = model.to(device)
model.eval()
all_true = []
all_preds = []

if torch.cuda.is_available():
    torch.cuda.empty_cache()

try:
    with torch.no_grad():
        for batch_X, batch_y in test_loader: 
            batch_X = batch_X.to(device, dtype=torch.float32)
            
            _, _, _, class_preds = model(batch_X)
            _, predicted = torch.max(class_preds, 1)
            
            all_true.extend(batch_y.numpy())
            all_preds.extend(predicted.cpu().numpy())

    accuracy = accuracy_score(all_true, all_preds)
    print(f"True Generalization Accuracy: {accuracy * 100:.2f}%\n")

    target_names = ["Left Hand", "Right Hand", "Both Feet", "Tongue"]
    print(classification_report(all_true, all_preds, target_names=target_names))
    print("Confusion Matrix:")
    print(confusion_matrix(all_true, all_preds))

except RuntimeError as e:
    print(f"\n CAUGHT THE EXACT ERROR:\n{e}")


--- FINAL EVALUATION ON UNSEEN DATA ---
True Generalization Accuracy: 29.31%

              precision    recall  f1-score   support

   Left Hand       0.22      0.27      0.24        15
  Right Hand       0.23      0.21      0.22        14
   Both Feet       0.50      0.53      0.52        15
      Tongue       0.18      0.14      0.16        14

    accuracy                           0.29        58
   macro avg       0.28      0.29      0.29        58
weighted avg       0.29      0.29      0.29        58

Confusion Matrix:
[[4 5 3 3]
 [6 3 1 4]
 [2 3 8 2]
 [6 2 4 2]]
